# Notebook 06 — Évaluation finale
## Phase 3 : Évaluation sur le test set et optimisation du seuil métier

Objectif : évaluer le modèle final sur le jeu de test intouché, comparer aux objectifs ML de Phase 1, et optimiser le seuil de décision à partir d’une matrice de coût asymétrique.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, precision_recall_curve, confusion_matrix)
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path('..').resolve()
DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
MODEL_DIR = PROJECT_ROOT / 'models'

validation_df = pd.read_csv(DATA_DIR / 'validation.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')

X_val = validation_df.drop(columns=['bad_nutrition'])
y_val = validation_df['bad_nutrition']
X_test = test_df.drop(columns=['bad_nutrition'])
y_test = test_df['bad_nutrition']

trained = joblib.load(MODEL_DIR / 'tuned_model.joblib')
model = trained['pipeline']

print('Validation shape:', X_val.shape)
print('Test shape:', X_test.shape)
print('Loaded tuned model:', type(model))

## 1. Évaluation sur le test set au seuil par défaut (0.5)

Nous calculons les métriques principales et complémentaires, puis affichons la matrice de confusion.

In [ ]:
y_prob_test = model.predict_proba(X_test)[:, 1]
y_pred_default = (y_prob_test >= 0.5).astype(int)

metrics_default = {
    'accuracy': accuracy_score(y_test, y_pred_default),
    'precision': precision_score(y_test, y_pred_default),
    'recall': recall_score(y_test, y_pred_default),
    'f1': f1_score(y_test, y_pred_default),
    'roc_auc': roc_auc_score(y_test, y_prob_test)
}
print('Metrics au seuil 0.5 :')
print(metrics_default)
print('Confusion matrix (default threshold):')
print(confusion_matrix(y_test, y_pred_default))

## 2. Optimisation du seuil de décision selon le coût métier

Nous utilisons le jeu de validation pour choisir le seuil qui minimise le coût total.
Remplacez `FN_COST` et `FP_COST` par les valeurs définies en Phase 1.

In [ ]:
FN_COST = 1000  # Modifier selon le coût métier de Phase 1
FP_COST = 50    # Modifier selon le coût métier de Phase 1

y_prob_val = model.predict_proba(X_val)[:, 1]
thresholds = np.arange(0.1, 0.91, 0.01)
records = []

for threshold in thresholds:
    y_pred_val = (y_prob_val >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, y_pred_val).ravel()
    cost = FN_COST * fn + FP_COST * fp
    records.append({
        'threshold': threshold,
        'precision': precision_score(y_val, y_pred_val),
        'recall': recall_score(y_val, y_pred_val),
        'f1': f1_score(y_val, y_pred_val),
        'cost': cost,
        'fp': fp,
        'fn': fn
    })

threshold_df = pd.DataFrame(records)
best_row = threshold_df.loc[threshold_df['cost'].idxmin()]
best_threshold = best_row['threshold']

print('Seuil optimal (validation) :', best_threshold)
print('Coût métier minimal :', best_row['cost'])
print('Precision:', best_row['precision'])
print('Recall:', best_row['recall'])
print('F1:', best_row['f1'])

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(threshold_df['threshold'], threshold_df['precision'], label='Precision')
plt.plot(threshold_df['threshold'], threshold_df['recall'], label='Recall')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Precision et Recall en fonction du seuil')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(threshold_df['threshold'], threshold_df['cost'], color='red')
plt.xlabel('Threshold')
plt.ylabel('Coût total')
plt.title('Coût métier estimé selon le seuil')
plt.grid(True)
plt.show()

## 3. Évaluation finale sur le test set au seuil optimal

Nous appliquons le seuil retenu sur le jeu de test pour mesurer l’impact réel et comparer au seuil par défaut.

In [ ]:
y_pred_test_opt = (y_prob_test >= best_threshold).astype(int)
metrics_opt = {
    'accuracy': accuracy_score(y_test, y_pred_test_opt),
    'precision': precision_score(y_test, y_pred_test_opt),
    'recall': recall_score(y_test, y_pred_test_opt),
    'f1': f1_score(y_test, y_pred_test_opt),
    'roc_auc': roc_auc_score(y_test, y_prob_test)
}
print('Metrics au seuil optimal :')
print(metrics_opt)
print('Confusion matrix (optimal threshold):')
print(confusion_matrix(y_test, y_pred_test_opt))

In [ ]:
plt.figure(figsize=(8, 6))
fpr, tpr, _ = roc_curve(y_test, y_prob_test)
plt.plot(fpr, tpr, label=f"ROC AUC = {metrics_default['roc_auc']:.3f}")
plt.plot([0, 1], [0, 1], '--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Courbe ROC')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 6))
precision, recall, _ = precision_recall_curve(y_test, y_prob_test)
plt.plot(recall, precision, label='Precision-Recall curve')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Courbe Precision-Recall')
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(y_prob_test[y_test == 0], color='green', label='Classe 0 (bon)', kde=True, stat='density', alpha=0.5)
sns.histplot(y_prob_test[y_test == 1], color='red', label='Classe 1 (mauvais)', kde=True, stat='density', alpha=0.5)
plt.xlabel('Probabilité prédite de bad_nutrition')
plt.title('Distribution des probabilités par classe réelle')
plt.legend()
plt.show()

In [ ]:
final_artifact = {
    'pipeline': model,
    'threshold': float(best_threshold),
    'cost_config': {'fn_cost': FN_COST, 'fp_cost': FP_COST},
    'metrics_default': metrics_default,
    'metrics_optimal': metrics_opt
}
joblib.dump(final_artifact, MODEL_DIR / 'final_model.joblib')
print('Final model artifact saved to models/final_model.joblib')

## 4. Synthèse finale

- Le modèle final est évalué sur le jeu de test intouché.
- Le seuil métier optimal est choisi sur le jeu de validation.
- Le fichier `models/final_model.joblib` contient le pipeline tuné et le seuil optimal en métadonnées.
- Remplacez `FN_COST` et `FP_COST` par les valeurs définies en Phase 1 pour obtenir une optimisation véritablement métier.